In [10]:
import math
from collections import Counter

# 1. Dataset parsing (from the provided CSV data)
headers = ['Age', 'Income', 'student', 'credit']
dataset = [
    ['youth', 'high', 'no', 'fair', 'no'],
    ['youth', 'high', 'no', 'excellent', 'no'],
    ['middle', 'high', 'no', 'fair', 'yes'],
    ['senior', 'medium', 'no', 'fair', 'yes'],
    ['senior', 'low', 'yes', 'fair', 'yes'],
    ['senior', 'low', 'yes', 'excellent', 'no'],
    ['middle', 'low', 'yes', 'excellent', 'yes'],
    ['youth', 'medium', 'no', 'fair', 'no'],
    ['youth', 'low', 'yes', 'fair', 'yes'],
    ['senior', 'medium', 'yes', 'fair', 'yes'],
    ['youth', 'medium', 'yes', 'excellent', 'yes'],
    ['middle', 'medium', 'no', 'excellent', 'yes'],
    ['middle', 'high', 'yes', 'fair', 'yes'],
    ['senior', 'medium', 'no', 'excellent', 'no']
]

# 2. Mathematical Helper Functions
def get_entropy(labels):
    total = len(labels)
    return -sum((count / total) * math.log2(count / total) for count in Counter(labels).values())

def get_gini(labels):
    total = len(labels)
    return 1.0 - sum((count / total) ** 2 for count in Counter(labels).values())

# 3. Attribute Selection Measures
def evaluate_split(data, attr_idx, method='id3'):
    total_len = len(data)
    labels = [row[-1] for row in data]
    
    # Partition data by unique values of the chosen attribute
    partitions = {}
    for row in data:
        partitions.setdefault(row[attr_idx], []).append(row)
        
    if method == 'cart':
        # Minimize Gini Index
        gini_split = sum((len(part) / total_len) * get_gini([r[-1] for r in part]) for part in partitions.values())
        return -gini_split # Return negative because we want to maximize this metric
        
    # For ID3 and C4.5, compute Information Gain
    base_entropy = get_entropy(labels)
    cond_entropy = sum((len(part) / total_len) * get_entropy([r[-1] for r in part]) for part in partitions.values())
    info_gain = base_entropy - cond_entropy
    
    if method == 'id3':
        return info_gain
    
    if method == 'c45':
        # Split Info (Intrinsic Value) to normalize gain
        split_info = -sum((len(part) / total_len) * math.log2(len(part) / total_len) for part in partitions.values())
        return info_gain / split_info if split_info != 0 else 0

# 4. Decision Tree Builder
def build_tree(data, features, method='id3'):
    labels = [row[-1] for row in data]
    if len(set(labels)) == 1:
        return labels[0]
    if not features:
        return Counter(labels).most_common(1)[0][0]
    
    # Find best split attribute
    best_gain = -float('inf')
    best_attr = None
    for f_idx in features:
        gain = evaluate_split(data, f_idx, method)
        if gain > best_gain:
            best_gain, best_attr = gain, f_idx
            
    if best_gain <= 0 and method != 'cart': 
        return Counter(labels).most_common(1)[0][0]
        
    # Build branches
    tree = {headers[best_attr]: {}}
    remaining_features = [f for f in features if f != best_attr]
    
    partitions = {}
    for row in data:
        partitions.setdefault(row[best_attr], []).append(row)
        
    for val, part in partitions.items():
        tree[headers[best_attr]][val] = build_tree(part, remaining_features, method)
        
    return tree

# 5. Predict function
def predict(tree, instance):
    if not isinstance(tree, dict):
        return tree
    root = list(tree.keys())[0]
    val = instance[headers.index(root)]
    sub_tree = tree[root].get(val)
    if sub_tree is None: # Fallback if branch is missing for test case
        return None
    return predict(sub_tree, instance)

# 6. Evaluation Metrics Helper
def evaluate_model(tree, test_data):
    correct = 0
    for row in test_data:
        pred = predict(tree, row[:-1])
        if pred == row[-1]:
            correct += 1
    return correct / len(test_data)

# Run and show results for ID3, C4.5, and CART
print("--- DECISION TREES GENERATED ---")
for method in ['id3', 'c45', 'cart']:
    # Train and test on the same dataset for simplicity 
    features_indices = list(range(len(headers)))
    tree = build_tree(dataset, features_indices, method=method)
    accuracy = evaluate_model(tree, dataset)
    
    print(f"\n[{method.upper()} Algorithm]")
    print(f"Tree Structure: {tree}")
    print(f"Training Accuracy: {accuracy * 100:.1f}%")

--- DECISION TREES GENERATED ---

[ID3 Algorithm]
Tree Structure: {'Age': {'youth': {'student': {'no': 'no', 'yes': 'yes'}}, 'middle': 'yes', 'senior': {'credit': {'fair': 'yes', 'excellent': 'no'}}}}
Training Accuracy: 100.0%

[C45 Algorithm]
Tree Structure: {'Age': {'youth': {'student': {'no': 'no', 'yes': 'yes'}}, 'middle': 'yes', 'senior': {'credit': {'fair': 'yes', 'excellent': 'no'}}}}
Training Accuracy: 100.0%

[CART Algorithm]
Tree Structure: {'Age': {'youth': {'student': {'no': 'no', 'yes': 'yes'}}, 'middle': 'yes', 'senior': {'credit': {'fair': 'yes', 'excellent': 'no'}}}}
Training Accuracy: 100.0%
